# 최근 K개 turn만 모델에 전달하기

> 업데이트 기준: 2026-09 · LangChain 1.x / LangGraph 1.x

과거의 `ConversationBufferWindowMemory(k=...)` 대신, 현재는 다음 두 책임을 분리하는
편이 안전합니다.

1. checkpointer에는 전체 스레드 상태를 보존합니다.
2. 모델을 호출하기 직전에 필요한 최근 turn만 선택합니다.

이렇게 하면 감사·디버깅에 필요한 원본 상태는 유지하면서 모델 비용과 산만함을
줄일 수 있습니다. 여기서는 **완료된 최근 2개 turn + 현재 질문**만 모델에 전달합니다.


In [1]:
# 필요한 경우 아래 줄의 주석을 해제하고 한 번만 실행하세요.
# %pip install -qU "langchain>=1.0" "langchain-openai>=1.0" "langgraph>=1.0" python-dotenv


In [2]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 다른 공급자를 쓸 때는 예: anthropic:claude-... 처럼 지정할 수 있습니다.
MODEL_ID = os.getenv("CHAT_MODEL", "openai:gpt-5.6-luna")
model = init_chat_model(MODEL_ID)


In [3]:
from langchain.messages import SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph

K = 2
SYSTEM_PROMPT = "당신은 친절한 은행 상담 챗봇입니다."


def context_for_model(messages):
    """완료된 최근 K개 turn과 현재 사용자 메시지를 선택합니다."""
    if not messages or messages[-1].type != "human":
        raise ValueError("마지막 메시지는 현재 사용자의 human 메시지여야 합니다.")

    history, current_user_message = messages[:-1], messages[-1]
    recent_history = history[-2 * K :]  # human/ai 한 쌍이 1 turn
    return [
        SystemMessage(content=SYSTEM_PROMPT),
        *recent_history,
        current_user_message,
    ]


def call_model(state: MessagesState):
    response = model.invoke(context_for_model(state["messages"]))
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_edge(START, "model")
builder.add_edge("model", END)
window_chat = builder.compile(checkpointer=InMemorySaver())


## 여러 turn 실행하기

매 호출에는 새 메시지만 넘기지만, checkpointer가 같은 `thread_id`의 전체 메시지를
노드에 복원합니다.


In [4]:
config = {"configurable": {"thread_id": "window-demo"}}

questions = [
    "비대면 계좌 개설을 시작하려면 무엇이 필요한가요?",
    "신분증을 준비했습니다. 다음 단계는 무엇인가요?",
    "본인 인증까지 마쳤습니다. 이제 무엇을 하나요?",
    "이용 약관에도 동의했습니다. 마지막 단계는 무엇인가요?",
    "가장 최근 안내만 짧게 다시 말해 주세요.",
]

for question in questions:
    result = window_chat.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config,
    )
    print(f"Q: {question}\nA: {result['messages'][-1].content}\n")


Q: 비대면 계좌 개설을 시작하려면 무엇이 필요한가요?
A: 비대면 계좌 개설을 시작하려면 보통 다음이 필요합니다.

- **본인 명의 휴대전화**: 문자·앱 인증용
- **신분증**: 주민등록증 또는 운전면허증  
  - 외국인은 여권·외국인등록증 등 추가 서류가 필요할 수 있습니다.
- **본인 명의의 기존 은행계좌** 또는 **영상통화·공동인증서 등 본인확인 수단**
- **스마트폰과 은행 앱**
- **카메라·인터넷 연결**: 신분증 촬영 및 본인 확인
- **금융거래 목적 관련 정보**: 직업, 거래 목적, 자금 원천 등을 입력할 수 있습니다.

은행에 따라 개설 가능 시간, 추가 서류, 한도제한계좌 적용 여부가 다르므로 이용하려는 은행 앱에서 사전 안내를 확인해 주세요. 미성년자나 외국인은 별도 절차가 필요할 수 있습니다.

Q: 신분증을 준비했습니다. 다음 단계는 무엇인가요?
A: 신분증을 준비하셨다면 다음 순서로 진행하시면 됩니다.

1. **이용하려는 은행의 공식 앱을 설치하고 실행**합니다.  
   - 앱스토어·플레이스토어에서 은행명을 직접 검색해 설치하세요.
2. **‘비대면 계좌 개설’ 또는 ‘입출금 계좌 개설’** 메뉴를 선택합니다.
3. **휴대전화 본인인증**을 진행합니다.
4. 앱 안내에 따라 **신분증을 촬영**합니다.  
   - 빛 반사나 흔들림이 없도록 평평한 곳에서 촬영하세요.
5. **얼굴 촬영 또는 영상통화 등 추가 본인확인**을 진행할 수 있습니다.
6. **개인정보, 직업, 거래 목적, 자금 원천** 등을 입력합니다.
7. 필요하면 **본인 명의의 기존 계좌로 1원 인증** 또는 다른 인증 절차를 진행합니다.
8. 약관에 동의하고 **계좌 비밀번호·이체한도** 등을 설정하면 개설이 완료됩니다.

신분증 사진, 인증번호, 계좌 비밀번호는 누구에게도 공유하지 마시고, 반드시 공식 은행 앱에서만 진행하세요. 은행에 따라 절차와 추가 서류가 다를 수 있습니다.

Q: 본인 인증까지 마쳤습니다. 이제 무엇을 하나요?
A: 

## 저장된 전체 기록과 실제 모델 입력 비교

아래에서 checkpoint에는 모든 turn이 남아 있지만, 마지막 호출에 전달된 컨텍스트는
제한되어 있음을 확인할 수 있습니다.


In [5]:
all_messages = window_chat.get_state(config).values["messages"]
last_call_context = context_for_model(all_messages[:-1])  # 마지막 AI 응답 직전 상태

print("checkpoint에 저장된 메시지 수:", len(all_messages))
print("마지막 모델 호출에 전달된 메시지 수:", len(last_call_context))
print()

for message in last_call_context:
    print(f"[{message.type}] {message.content}")


checkpoint에 저장된 메시지 수: 10
마지막 모델 호출에 전달된 메시지 수: 6

[system] 당신은 친절한 은행 상담 챗봇입니다.
[human] 본인 인증까지 마쳤습니다. 이제 무엇을 하나요?
[ai] 본인 인증을 마치셨다면 보통 다음 단계로 진행합니다.

1. **계좌 종류 선택**  
   입출금통장, 적금 등 원하는 상품을 선택합니다.

2. **개인정보 및 거래 정보 입력**  
   주소, 직업, 거래 목적, 자금 원천 등을 입력합니다.

3. **약관과 상품설명서 확인**  
   내용을 읽고 동의합니다. 금리, 수수료, 이체한도도 확인하세요.

4. **계좌 비밀번호와 이체한도 설정**  
   다른 서비스에서 사용하지 않는 비밀번호를 설정합니다.

5. **추가 인증 진행**  
   필요하면 본인 명의 기존 계좌를 이용한 1원 인증 등을 진행합니다.

6. **신청 내용 확인 후 개설 완료**  
   완료 화면에서 계좌번호와 거래 가능 여부를 확인하고, 안내에 따라 체크카드나 모바일뱅킹을 신청할 수 있습니다.

은행에 따라 **한도제한계좌**로 개설될 수 있으며, 이후 거래 목적을 증빙하면 한도 해제가 가능합니다.
[human] 이용 약관에도 동의했습니다. 마지막 단계는 무엇인가요?
[ai] 이용 약관에 동의하셨다면 보통 **신청 내용 최종 확인 및 계좌 개설 완료** 단계입니다.

1. 입력한 개인정보와 상품 조건을 다시 확인합니다.  
2. **계좌 비밀번호와 이체 비밀번호**를 설정합니다.  
3. 필요하면 본인 명의 계좌를 통한 추가 인증을 진행합니다.  
4. **‘신청하기’ 또는 ‘계좌 개설 완료’** 버튼을 누릅니다.  
5. 완료 화면에서 계좌번호, 한도, 거래 가능 여부를 확인합니다.

계좌번호와 비밀번호를 저장하거나 공유할 때는 주의하시고, 반드시 공식 은행 앱의 완료 화면에서 확인하세요. 완료 후에는 모바일뱅킹 등록이나 체크카드 신청을 선택할 수 있습니다.
[human] 가장 최근 안내만 짧게 다시 말해 주세

상태에서도 오래된 메시지를 영구 삭제해야 한다면 `RemoveMessage`를 반환하는 별도
노드를 둘 수 있습니다. 다만 단순한 컨텍스트 절약 목적이라면 위처럼 **저장과 모델 입력을
분리**하는 방식이 보통 더 유연합니다.
